# Storage durability checks — M2-1, M2-14

Both cases are about `server/callbacks/utils/json_file_store.py` staying correct under conditions a normal
single-tester CLI session never exercises: two writers at once, and a crash mid-write. Both are naturally
suited to real-code testing — a human can't reliably time a `kill -9` to land mid-`write()`, but a notebook
can simulate the on-disk aftermath exactly and check the invariant directly.

**This notebook found a real bug while being built** (see the M2-14 section below) — kept in, not edited
out, because it's the clearest demonstration yet of why "test against the real code" beats "read the code
and trust the reasoning."

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "server").exists():
    REPO_ROOT = Path("__file__").resolve().parents[2] if Path("__file__").exists() else Path.cwd().parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo root on sys.path:", REPO_ROOT)
assert (REPO_ROOT / "server" / "callbacks").exists(), (
    "Couldn't find server/callbacks/ from here -- open this notebook with the repo root as the "
    "Jupyter working directory, or edit REPO_ROOT above by hand."
)

import harness


---
## M2-1 — two large concurrent writes must not interleave/corrupt the file

**Real-world scenario:** two large, genuinely simultaneous consent-notify POSTs land on the server at
close to the same instant (this server runs threaded via `asyncio.to_thread()` for every blocking file
write, so two in-flight requests really can call `_append()` on `consents.jsonl` concurrently). Python's
`open(path, "a")` gives no atomicity guarantee for a write larger than the OS pipe buffer — without a lock,
two big writes landing at the same moment could interleave their bytes mid-line, corrupting BOTH consent
records, not just delaying one of them.

**Fix:** a per-file `threading.Lock` in `_append()` serializes every write to a given file from within this
one process.

**Pass criteria:** after 20 large concurrent writes, every line in the file is still valid, unmangled JSON,
and all 20 records are independently correct.

In [ ]:
import threading
import json as jsonmod

from server.callbacks.repository.consent_repository import save_consent, get_all_consents

scratch = harness.activate_scratch_storage("m2_1")

N = 20
def writer(i):
    save_consent(f"consent-concurrent-{i}", {
        "care_contexts": [{"referenceNumber": f"cc-{j}"} for j in range(30)],
        "padding": "x" * 500,  # inflate each line's size to make interleaving more likely if unlocked
    })

threads = [threading.Thread(target=writer, args=(i,)) for i in range(N)]
for t in threads:
    t.start()
for t in threads:
    t.join()

lines = (scratch / "consents.jsonl").read_text().splitlines()
all_parse = True
for ln in lines:
    try:
        jsonmod.loads(ln)
    except ValueError:
        all_parse = False

harness.check(f"all {len(lines)} lines are valid, unmangled JSON after {N} concurrent writers", all_parse and len(lines) == N)
harness.check("all 20 consents are independently retrievable and correct", len(get_all_consents()) == N)


---
## M2-14 — a crash mid-write must not corrupt the NEXT record too

**Original claim (Batch 2 runbook, "no fix written"):** "the append-only storage design means a crash
mid-write can only ever leave one partial trailing line, which `_replay()` already skips on read. There
should be no 'next record' to corrupt, since nothing is written after the in-flight line."

**What this notebook found:** that reasoning has a gap. `_append()`'s write is
`f.write(json.dumps(record) + "\n")` as one call — if the process is killed before the trailing `"\n"`
reaches disk, the file's last line has NO newline at the end. The very next `_append()` call (e.g. the
server's first write right after restarting) opens the file in append mode and writes straight onto the
end of that existing content — landing on the SAME physical line as the stale partial data, with nothing
separating them. `_replay()` reads that combined line, `json.loads()` fails on the whole thing, and the
fresh, otherwise-perfectly-good record written after restart is silently lost too — not just the one that
was genuinely in-flight during the crash.

**The fix** (applied directly in this session as a result of this finding, in
`server/callbacks/utils/json_file_store.py`): before writing, `_append()` now checks whether the file
already ends with a newline (or is empty/doesn't exist yet) and, if not, writes a leading `"\n"` first —
isolating any stale partial line onto its own (still skipped, but now harmless) line so a fresh write can
never be dragged into it.

**Pass criteria:** a fully-written record before the "crash" survives; the crash's own partial line is
skipped; a fresh write made right after (simulating a post-restart write) is intact and correctly
readable — separately from the corrupted line, not merged into it.

In [ ]:
from server.callbacks.utils.json_file_store import set_key, get_key, get_all

scratch2 = harness.activate_scratch_storage("m2_14")

set_key("crash_test.jsonl", "good-record-1", {"value": "intact"})

# Simulate a kill -9 mid-write: append a truncated/garbled trailing line directly to the file
# (bypassing set_key, which always writes a complete, newline-terminated line) -- this is what a
# crash mid f.write(json.dumps(record) + "\n") can leave behind if the "\n" never made it to disk.
with open(scratch2 / "crash_test.jsonl", "a", encoding="utf-8") as f:
    f.write('{"key": "in-flight-record", "value": {"partial": "dat')  # deliberately cut off, NO trailing newline

try:
    state_after_crash = get_all("crash_test.jsonl")
    read_ok = True
except Exception:
    read_ok = False
    state_after_crash = {}

harness.check("read right after the simulated crash doesn't raise", read_ok)
harness.check("the earlier, fully-written record survives intact", state_after_crash.get("good-record-1") == {"value": "intact"})
harness.check("the partial/garbled in-flight line is skipped, not returned", "in-flight-record" not in state_after_crash)

# Now simulate the server restarting and making its first fresh write.
set_key("crash_test.jsonl", "fresh-record-after-restart", {"value": "post-restart-write"})

harness.check(
    "a fresh write right after the crash is stored and reads back correctly (THIS is the check that used to FAIL before the fix)",
    get_key("crash_test.jsonl", "fresh-record-after-restart") == {"value": "post-restart-write"},
)
